# FullPipelineTrigger — Classificazione single-cell RNA-seq

Esegue in sequenza i notebook del progetto tramite `nbconvert --execute --inplace`.
Ogni notebook viene eseguito e i suoi output vengono salvati al suo interno, così
aprendolo si vedono i risultati in linea.

**Ordine di esecuzione:** `data_loader.ipynb` -> `EDA.ipynb` -> `Machine_learning_models.ipynb`

**Requisiti:** kernel del `venv` del progetto (con `ipykernel` e `nbconvert` installati)
e i CSV presenti in `data_train/` e `data_test/`.

## 1 · Configurazione

In [3]:
# Modalita di esecuzione: True = timeout ridotto (prove rapide),
# False = timeout ampio (esecuzione completa).
FAST_MODE = True

# Notebook da eseguire, in ordine. Per aggiungere o togliere un passo,
# modifica questa lista (i nomi sono relativi alla cartella notebooks/).
NOTEBOOKS = [
    "data_loader.ipynb",
    "EDA.ipynb",
    "Machine_learning_models.ipynb",
]

# Descrizione breve di ogni notebook (una riga), mostrata prima dell'esecuzione.
DESCRIZIONI = {
    "data_loader.ipynb": "Caricamento dei dati e anteprima (shape, prime righe).",
    "EDA.ipynb": "Analisi esplorativa: struttura, classi, geni discriminanti, PCA.",
    "Machine_learning_models.ipynb": "Addestramento e valutazione dei classificatori.",
}

## 2 · Verifica dei prerequisiti

In [ ]:
import os
import sys
import time
import subprocess
from pathlib import Path


def trova_radice(start=None):
    """Risale le cartelle fino a trovare src/setup.py (radice del progetto)."""
    p = (start or Path.cwd()).resolve()
    for _ in range(6):
        if (p / "src" / "setup.py").is_file():
            return p
        if p.parent == p:
            break
        p = p.parent
    raise FileNotFoundError("Radice del progetto non trovata (manca src/setup.py).")


ROOT = trova_radice()
NB_DIR = ROOT / "notebooks"

# Controllo: i notebook esistono su disco.
problemi = []
for nb in NOTEBOOKS:
    if not (NB_DIR / nb).is_file():
        problemi.append(f"Notebook mancante: notebooks/{nb}")

# Controllo: i CSV richiesti sono presenti.
csv_richiesti = [
    ROOT / "data_train" / "X_train.csv",
    ROOT / "data_train" / "y_train.csv",
    ROOT / "data_test" / "X_test.csv",
    ROOT / "data_test" / "y_test.csv",
]
for csv in csv_richiesti:
    if not csv.is_file():
        problemi.append(f"File dati mancante: {csv.relative_to(ROOT)}")

if problemi:
    print("Prerequisiti NON soddisfatti:")
    for p in problemi:
        print("  -", p)
    raise SystemExit("Interruzione: risolvere i problemi elencati prima di continuare.")

print("Prerequisiti soddisfatti.")
print(f"Radice del progetto: {ROOT}")
print(f"Notebook da eseguire: {len(NOTEBOOKS)}")

Prerequisiti soddisfatti.
Radice del progetto: C:\Users\julie\Dev\GitHub\ML_SingleCell_Classification
Notebook da eseguire: 3


## 3 · Esecuzione dei notebook

In [5]:
# Timeout per cella: ridotto in FAST_MODE, ampio in esecuzione completa.
timeout_s = 1200 if FAST_MODE else 3600

# Ambiente: codifica UTF-8 e flag opzionale leggibile dai notebook con os.getenv.
env = os.environ.copy()
env["PYTHONIOENCODING"] = "utf-8"
env["SC_FAST_MODE"] = "1" if FAST_MODE else "0"

risultati = []
for nb in NOTEBOOKS:
    descrizione = DESCRIZIONI.get(nb, "(nessuna descrizione)")
    print("=" * 64)
    print(f"Notebook: {nb}")
    print(f"Cosa fa : {descrizione}")
    print("=" * 64)

    cmd = [
        sys.executable, "-m", "jupyter", "nbconvert",
        "--to", "notebook", "--execute", "--inplace",
        f"--ExecutePreprocessor.timeout={timeout_s}",
        str(NB_DIR / nb),
    ]

    inizio = time.time()
    esito = subprocess.run(
        cmd, cwd=str(ROOT), env=env,
        capture_output=True, text=True, encoding="utf-8", errors="replace",
    )
    durata = time.time() - inizio
    ok = esito.returncode == 0
    risultati.append({"notebook": nb, "ok": ok, "secondi": round(durata, 1)})

    if ok:
        print(f"Completato in {durata:.1f} s.")
    else:
        # Non interrompiamo: segnaliamo l'errore e proseguiamo con gli altri.
        print(f"FALLITO dopo {durata:.1f} s (exit code {esito.returncode}).")
        errore = (esito.stderr or "").strip().splitlines()
        if errore:
            print("Ultime righe dell'errore:")
            for riga in errore[-15:]:
                print("   ", riga)
    print()

Notebook: data_loader.ipynb
Cosa fa : Caricamento dei dati e anteprima (shape, prime righe).
Completato in 49.6 s.

Notebook: EDA.ipynb
Cosa fa : Analisi esplorativa: struttura, classi, geni discriminanti, PCA.
Completato in 64.1 s.

Notebook: Machine_learning_models.ipynb
Cosa fa : Addestramento e valutazione dei classificatori.
Completato in 455.6 s.



## 4 · Riepilogo finale

In [ ]:
# Tabella: stato (OK / X) e tempo per ogni notebook.
print("Riepilogo esecuzione")
print("-" * 56)
print(f"{'Notebook':36s} {'Stato':6s} {'Tempo (s)':>10s}")
print("-" * 56)
for r in risultati:
    stato = "✓" if r["ok"] else "✗"
    print(f"{r['notebook']:36s} {stato:6s} {r['secondi']:>10.1f}")
print("-" * 56)

n_ok = sum(1 for r in risultati if r["ok"])
print(f"Completati con successo: {n_ok}/{len(risultati)}")
if n_ok < len(risultati):
    falliti = [r["notebook"] for r in risultati if not r["ok"]]
    print("Notebook falliti:", ", ".join(falliti))
    print("Aprire il notebook fallito per ispezionare l'errore in dettaglio.")
else:
    print("Pipeline completata: tutti i notebook eseguiti correttamente.")

Riepilogo esecuzione
--------------------------------------------------------
Notebook                             Stato   Tempo (s)
--------------------------------------------------------
data_loader.ipynb                    ✓            49.6
EDA.ipynb                            ✓            64.1
Machine_learning_models.ipynb        ✓           455.6
--------------------------------------------------------
Completati con successo: 3/3
Pipeline completata: tutti i notebook eseguiti correttamente.
